# Spatial Correctness and Joint Weather vs Bird Richness

This notebook analyzes bird richness with weather variables jointly (not isolated one-by-one): temp_mean, rainfall, wind_mean, humid_mean, shortwave_radiation.

Scope:
- Spatial thinning: one record per (district, 5 km cell, species)
- Cell-level bird richness
- Joint multivariate weather-richness model
- District-level weather profile and richness patterns


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 120)
plt.rcParams["figure.dpi"] = 120
RANDOM_SEED = 42


## 1. Load bird + weather columns

In [ ]:
candidate_paths = [
    Path("file6.csv"),
    Path("../file6.csv"),
    Path("../../file6.csv"),
    Path("final5.csv"),
    Path("../final5.csv"),
    Path("../../final5.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find file6.csv or final5.csv from this notebook location.")

weather_cols = [
    "temp_mean",
    "rainfall",
    "wind_mean",
    "humid_mean",
    "shortwave_radiation",
]

raw = pd.read_csv(data_path, low_memory=False)
required_cols = [
    "stateProvince",
    "verbatimScientificName",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
] + weather_cols

missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw[required_cols].copy()
df["eventDate"] = pd.to_datetime(df["eventDate"], errors="coerce")
for c in ["decimalLatitude", "decimalLongitude"] + weather_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["stateProvince"] = df["stateProvince"].astype(str).str.strip()
df["verbatimScientificName"] = df["verbatimScientificName"].astype(str).str.strip()

df = df.dropna(subset=[
    "stateProvince",
    "verbatimScientificName",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
] + weather_cols)

print(f"Using: {data_path.resolve()}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Districts: {df['stateProvince'].nunique():,}")
print(f"Species: {df['verbatimScientificName'].nunique():,}")


## 2. Spatial thinning (district, 5 km cell, species)

In [ ]:
def add_5km_grid(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    lon0 = float(out["decimalLongitude"].median())
    lat0 = float(out["decimalLatitude"].median())
    lat_rad = np.radians(out["decimalLatitude"].to_numpy())
    out["x_km"] = (out["decimalLongitude"].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
    out["y_km"] = (out["decimalLatitude"].to_numpy() - lat0) * 110.574
    out["grid_x"] = np.floor(out["x_km"] / 5.0).astype(int)
    out["grid_y"] = np.floor(out["y_km"] / 5.0).astype(int)
    out["cell_id"] = (
        out["stateProvince"].astype(str)
        + "|"
        + out["grid_x"].astype(str)
        + "_"
        + out["grid_y"].astype(str)
    )
    return out

def thin_per_species_cell(frame: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    work = frame.copy()
    work["_thin_key"] = (
        work["stateProvince"].astype(str)
        + "|"
        + work["grid_x"].astype(str)
        + "_"
        + work["grid_y"].astype(str)
        + "|"
        + work["verbatimScientificName"].astype(str)
    )
    work["_rand"] = rng.random(len(work))
    return (
        work.sort_values("_rand")
        .groupby("_thin_key", as_index=False)
        .head(1)
        .drop(columns=["_thin_key", "_rand"])
    )

df_grid = add_5km_grid(df)
df_thin = thin_per_species_cell(df_grid, seed=RANDOM_SEED)

print(f"Records before thinning: {len(df_grid):,}")
print(f"Records after thinning:  {len(df_thin):,}")
print(f"Retained fraction:       {len(df_thin)/len(df_grid):.2%}")


## 3. Cell-level richness + joint weather table

In [ ]:
agg_dict = {"verbatimScientificName": "nunique"}
for c in weather_cols:
    agg_dict[c] = "mean"

cell_summary = (
    df_thin.groupby(["stateProvince", "cell_id"], as_index=False)
    .agg(agg_dict)
    .rename(columns={"verbatimScientificName": "species_richness"})
)

cell_summary = cell_summary.dropna(subset=["species_richness"] + weather_cols)

print(f"Cell rows for joint weather analysis: {len(cell_summary):,}")
cell_summary.head(10)


## 4. Joint multivariate model: richness ~ all weather variables

In [ ]:
model_df = cell_summary.copy()

for c in weather_cols:
    mu = model_df[c].mean()
    sd = model_df[c].std(ddof=0)
    model_df[f"z_{c}"] = (model_df[c] - mu) / (sd if sd > 0 else 1.0)

z_cols = [f"z_{c}" for c in weather_cols]
formula = "species_richness ~ " + " + ".join(z_cols)
fit = smf.ols(formula, data=model_df).fit()

print(fit.summary().tables[0])
print(fit.summary().tables[1])

rho, pval = stats.spearmanr(model_df["species_richness"], fit.fittedvalues)
print(f"Spearman(observed richness, fitted richness): r={rho:.3f}, p={pval:.3g}")


In [ ]:
coef_table = (
    fit.params.rename("coef").to_frame()
    .join(fit.pvalues.rename("p_value"))
    .reset_index()
    .rename(columns={"index": "term"})
)
coef_plot = coef_table[coef_table["term"].str.startswith("z_")].copy()
coef_plot = coef_plot.sort_values("coef")

plt.figure(figsize=(8.8, 5.4))
colors = ["#b03a2e" if x < 0 else "#2874a6" for x in coef_plot["coef"]]
plt.barh(coef_plot["term"], coef_plot["coef"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Joint model coefficients (standardized weather predictors)")
plt.xlabel("Coefficient on species richness")
plt.tight_layout()
plt.show()

coef_plot


## 5. Joint weather-richness structure visualizations

In [ ]:
corr_df = cell_summary[["species_richness"] + weather_cols].corr(method="spearman")

plt.figure(figsize=(8.6, 6.2))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, linewidths=0.4)
plt.title("Spearman correlations: richness and weather variables")
plt.tight_layout()
plt.show()


In [ ]:
pair_cols = ["species_richness"] + weather_cols
sample_n = min(3500, len(cell_summary))
pair_data = cell_summary[pair_cols].sample(sample_n, random_state=RANDOM_SEED)

g = sns.pairplot(
    pair_data,
    vars=pair_cols,
    corner=True,
    plot_kws={"alpha": 0.18, "s": 12, "color": "#2c3e50"},
    diag_kws={"bins": 24, "color": "#5d6d7e"},
)
g.fig.suptitle("Joint pairwise structure: richness + all weather variables", y=1.01)
plt.show()


## 6. District-level joint weather profile

In [ ]:
district_joint = (
    cell_summary.groupby("stateProvince", as_index=False)
    .agg({"species_richness": "mean", **{c: "mean" for c in weather_cols}, "cell_id": "count"})
    .rename(columns={"species_richness": "mean_cell_richness", "cell_id": "n_cells"})
)

for c in weather_cols:
    district_joint[f"z_{c}"] = (district_joint[c] - district_joint[c].mean()) / district_joint[c].std(ddof=0)

z_weather_cols = [f"z_{c}" for c in weather_cols]
profile = district_joint[["stateProvince"] + z_weather_cols].set_index("stateProvince")

display(district_joint.sort_values("mean_cell_richness", ascending=False).head(30))

plt.figure(figsize=(11.8, 6.6))
sns.heatmap(profile, cmap="coolwarm", center=0, linewidths=0.4)
plt.title("District joint weather profile (z-scored means)")
plt.xlabel("Weather variables (z-score)")
plt.ylabel("District")
plt.tight_layout()
plt.show()


In [ ]:
district_joint["pred_richness_from_weather"] = fit.predict(
    district_joint.assign(**{f"z_{c}": district_joint[f"z_{c}"] for c in weather_cols})
)

plt.figure(figsize=(7.8, 5.6))
sns.regplot(
    data=district_joint,
    x="pred_richness_from_weather",
    y="mean_cell_richness",
    scatter_kws={"s": 60, "alpha": 0.8, "color": "#1f618d"},
    line_kws={"color": "#c0392b"},
)
plt.title("District mean richness vs model-predicted richness from joint weather")
plt.xlabel("Predicted richness from joint weather model")
plt.ylabel("Observed mean cell richness")
plt.tight_layout()
plt.show()


## 7. Optional exports

In [ ]:
# Uncomment to export outputs.
# cell_summary.to_csv("weather_cell_summary.csv", index=False)
# district_joint.to_csv("weather_district_joint_summary.csv", index=False)
# coef_plot.to_csv("weather_joint_model_coefficients.csv", index=False)
